# Width-256 FineWeb optimizer sweep
Select a GPU runtime (A100 if available), then run all cells. Authorize Drive when prompted. The default runs NS5, PE3 and PE5 for seeds 24–26. Change FAMILY to "tikh" for the four exact Tikhonov settings. Both suites use the tokenizer embedded from the laptop experiment. Results are saved in My Drive/cs182-w256-colab. Use separate directories for runs from different devices. The captures alone use roughly 108 MiB per run, plus checkpoints and other files.

In [ ]:
FAMILY = "ns_pe"  # "ns_pe" or "tikh"
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
from pathlib import Path
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
ROOT = Path("/content/cs182-w256")
OUT = Path("/content/drive/MyDrive/cs182-w256-colab")
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select a BF16-capable GPU runtime."
print(torch.cuda.get_device_name())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets==5.2.0", "huggingface_hub==2.4.0", "tiktoken==0.14.0", "tokenizers==0.23.3", "matplotlib"])


In [ ]:
# @title Restore the embedded project files and fixed tokenizer
import base64, hashlib, io, zipfile
archive = base64.b64decode("...")
assert hashlib.sha256(archive).hexdigest() == "31f274ba3b78b0b6ba5a4433c39110a7afafb998e8da77919877fb9987f642d5"
with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
    bundle.extractall(ROOT)


In [ ]:
# Run the repository launcher in a fresh process using Colab's CUDA device.
runner = """
import os, sys
from pathlib import Path
root, out, family = sys.argv[1:]
sys.path.insert(0, str(Path(root) / "scripts"))
import run_w256 as suite
suite.OUT = Path(out)
conditions = {"ns_pe": suite.NS_PE, "tikh": suite.TIKH}[family]
original_train = suite.train
def train_gpu(args):
    original_train([*args, "--device", "cuda"])
suite.train = train_gpu
suite.run(conditions, create_tokenizer=False)
"""
subprocess.run([sys.executable, "-u", "-c", runner, str(ROOT), str(OUT), FAMILY], check=True)


Completed runs are reused when you rerun the notebook with the same Drive directory. If a disconnect leaves a run directory without results.json, rename that incomplete directory before retrying. Each completed run contains results.json, checkpoint.pt, momenta.pt, and transfer.png. Compare train_sha256 within seed and validation_sha256 across runs before combining results from different machines.